## Load Modules
- PyTorch as our preferred framework
- Torchvision as the pretrained model which is using ResNet
- cv2 for video processing
- Numpy to work with numpy arrays
- Plotly for plotting
- Pandas for data processing

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import numpy as np

from torchvision.models import ResNet18_Weights

In [2]:
import plotly.express as px
import pandas as pd

## Set Seed for Reproducibility

In [3]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

## Defining the ResNet VAE

In [4]:
class ResNetVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(ResNetVAE, self).__init__()

        # Load Pretrained ResNet18 as Encoder
        resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])  # Remove classifier

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Latent space
        self.fc_mu = nn.Linear(512, latent_dim)
        self.fc_logvar = nn.Linear(512, latent_dim)

        # Decoder: Convert latent vector back to image
        self.fc_decode = nn.Linear(latent_dim, 512 * 7 * 7)  # Upscale latent space

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()  # Normalize output to [0,1] range
        )

    def encode(self, x):
        x = self.encoder(x)  # Output shape: (batch, 512, 7, 7)
        x = self.global_pool(x)  # (batch, 512, 1, 1)
        x = torch.flatten(x, start_dim=1)  # (batch, 512)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.fc_decode(z)  # Expand latent space
        x = x.view(-1, 512, 7, 7)  # Reshape into feature map
        x = self.decoder(x)  # Decode to (3, 224, 224)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)  # Get reconstructed frame
        return z, mu, logvar, recon_x


In [5]:
## Defining Method for Video Processing

def preprocess_frame(frame):
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),  # Resize for ResNet
        transforms.ToTensor(),
    ])
    return transform(frame).unsqueeze(0)  # Add batch dimension

def extract_latent_from_video(video_path, model, device='cuda'):
    cap = cv2.VideoCapture(video_path)
    model.to(device)
    model.eval()

    latent_vectors = []
    reconstructed_frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = preprocess_frame(frame).to(device)
        with torch.no_grad():
            z, mu, logvar, recon_frame = model(frame)  # Now includes reconstruction
            latent_vectors.append(z.cpu().numpy().flatten())

            recon_frame = recon_frame.squeeze(0).cpu().permute(1, 2, 0).numpy()  # Ensure correct shape
            reconstructed_frames.append(recon_frame)

    cap.release()
    return np.array(latent_vectors), np.array(reconstructed_frames)

def save_reconstructed_video(reconstructed_frames, output_path, fps=30):
    height, width, _ = reconstructed_frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    for frame in reconstructed_frames:
        frame = (frame * 255).astype(np.uint8)  # Convert from [0,1] to [0,255]
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    
    out.release()

## Reconstruct Video to test Model (Trial)

In [ ]:
# Video for Reconstruction
# video_path = "banana_red.mp4"
# latent_vectors, reconstructed_frames = extract_latent_from_video(video_path, vae, device)

# Save Reconstructed Video
# save_reconstructed_video(reconstructed_frames, "reconstructed_banana_red.mp4")

# print("Reconstructed video saved as reconstructed_banana_red.mp4")

NameError: name 'vae' is not defined

## Process Video and Extract Latent Space

In [ ]:
def extract_latent_from_video(video_path, model, device='cuda'):
    cap = cv2.VideoCapture(video_path)
    model.to(device)
    model.eval()  # Set to eval mode

    latent_vectors = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = preprocess_frame(frame).to(device)
        with torch.no_grad():
            mu, logvar = model.encode(frame)
            latent_vector = model.reparameterize(mu, logvar)  # Get 2D latent representation
            latent_vectors.append(latent_vector.cpu().numpy().flatten())

    cap.release()
    return np.array(latent_vectors)  # Return array of latent vectors


## Run Model on a Video File

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vae = ResNetVAE(latent_dim=2).to(device)

In [ ]:
# Video 1
video_path = "misc\banana_red.mp4"
latent_vectors_1 = extract_latent_from_video(video_path, vae, device)

print("Latent representations shape:", latent_vectors_1.shape)

In [ ]:
# Video 2
video_path = "misc\banana_blue.mp4"
latent_vectors_2 = extract_latent_from_video(video_path, vae, device)

print("Latent representations shape:", latent_vectors_2.shape)

In [ ]:
# Video 3
video_path = "misc\banana_blue.mp4"
latent_vectors_3 = extract_latent_from_video(video_path, vae, device)

print("Latent representations shape:", latent_vectors_3.shape)

## Plotting Latent Spaces

In [ ]:
# Create a DataFrame for easier plotting
df1 = pd.DataFrame(latent_vectors_2, columns=["Latent X", "Latent Y"])
df1["Video"] = "Banana Blue"

df2 = pd.DataFrame(latent_vectors_1, columns=["Latent X", "Latent Y"])
df2["Video"] = "Banana Red"

# Combine both datasets
df = pd.concat([df1, df2])

# Plot with Plotly
fig = px.scatter(df, x="Latent X", y="Latent Y", color="Video", 
                 title="Latent Space Representation of Video Frames Comparing Videoes of Bananas Falling With Different Background Color",
                 opacity=0.7)

fig.show()

In [ ]:
# Create a DataFrame for easier plotting
df1 = pd.DataFrame(latent_vectors_2, columns=["Latent X", "Latent Y"])
df1["Video"] = "Banana Blue"

df2 = pd.DataFrame(latent_vectors_3, columns=["Latent X", "Latent Y"])
df2["Video"] = "Banana Blue Copy"

# Combine both datasets
df = pd.concat([df1, df2])

# Plot with Plotly
fig = px.scatter(df, x="Latent X", y="Latent Y", color="Video", 
                 title="Latent Space Representation of Video Frames Comparing Videoes of Bananas Falling of the Same Background Color",
                 opacity=0.7)

fig.show()

In [ ]:
# Video 4 Entirely Different Video of Puppies
video_path = "misc\puppies.mp4"
latent_vectors_4 = extract_latent_from_video(video_path, vae, device)

print("Latent representations shape:", latent_vectors_4.shape)

In [ ]:
# Create a DataFrame for easier plotting
df1 = pd.DataFrame(latent_vectors_2, columns=["Latent X", "Latent Y"])
df1["Video"] = "Banana Blue"

df2 = pd.DataFrame(latent_vectors_4, columns=["Latent X", "Latent Y"])
df2["Video"] = "Puppies"

# Combine both datasets
df = pd.concat([df1, df2])

# Plot with Plotly
fig = px.scatter(df, x="Latent X", y="Latent Y", color="Video", 
                 title="Latent Space Representation of Video Frames Comparing A Video of Bananas Falling VS A Video of 2 Puppies",
                 opacity=0.7)

fig.show()